In [1]:
from pathlib import Path
import random
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [3]:
physical_devices = tf.config.list_physical_devices()

print("Available devices:")

for device in physical_devices:
    print(device)

Available devices:
PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')


In [4]:
print(
    "Graphics processing units detected:",
    tf.config.list_physical_devices("GPU")
)

Graphics processing units detected: []


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("..")

TRAIN_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train"
)

VALIDATION_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "val"
)

MODELS_DIRECTORY = (
    PROJECT_ROOT
    / "models"
)

OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "hyperparameter_training"
)

HISTORY_DIRECTORY = (
    PROJECT_ROOT
    / "histories"
    / "hyperparameter_training"
)

In [8]:
if not TRAIN_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Training folder was not found:\n"
        f"{TRAIN_DIRECTORY.resolve()}"
    )

if not VALIDATION_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Validation folder was not found:\n"
        f"{VALIDATION_DIRECTORY.resolve()}"
    )

print("Training and validation folders were found.")

Training and validation folders were found.


In [9]:
training_class_folders = sorted([
    folder.name
    for folder in TRAIN_DIRECTORY.iterdir()
    if folder.is_dir()
])

validation_class_folders = sorted([
    folder.name
    for folder in VALIDATION_DIRECTORY.iterdir()
    if folder.is_dir()
])

print("Training class folders:")
print(training_class_folders)

print("\nValidation class folders:")
print(validation_class_folders)

Training class folders:
['fall', 'no_fall']

Validation class folders:
['fall', 'no_fall']


In [10]:
CLASS_NAMES = [
    "no_fall",
    "fall"
]

print("Class mapping:")

for class_number, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_number} = {class_name}"
    )

Class mapping:
0 = no_fall
1 = fall


In [11]:
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224

IMAGE_SIZE = (
    IMAGE_HEIGHT,
    IMAGE_WIDTH
)

CUSTOM_CNN_BATCH_SIZE = 32
MOBILENET_BATCH_SIZE = 32
RESNET_BATCH_SIZE = 16

CUSTOM_CNN_LEARNING_RATE = 0.001

TRANSFER_HEAD_LEARNING_RATE = 0.001
FINE_TUNING_LEARNING_RATE = 0.00001

CUSTOM_CNN_MAX_EPOCHS = 30

FROZEN_BACKBONE_MAX_EPOCHS = 15
FINE_TUNING_MAX_EPOCHS = 15

CUSTOM_CNN_DROPOUT = 0.40
MOBILENET_DROPOUT = 0.30
RESNET_DROPOUT = 0.40

L2_REGULARISATION = 0.0001

print("Hyperparameters created successfully.")

Hyperparameters created successfully.


In [12]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif"
}

def count_images(folder_path):
    return sum(
        1
        for file_path in folder_path.rglob("*")
        if (
            file_path.is_file()
            and file_path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    )

training_counts = {}

for class_name in CLASS_NAMES:
    class_path = (
        TRAIN_DIRECTORY
        / class_name
    )

    training_counts[class_name] = (
        count_images(class_path)
    )

print("Training image counts:")

for class_name, image_count in (
    training_counts.items()
):
    print(
        f"{class_name}: {image_count}"
    )

Training image counts:
no_fall: 684
fall: 439


In [13]:
training_labels_for_weights = []

for class_number, class_name in enumerate(
    CLASS_NAMES
):
    class_count = training_counts[
        class_name
    ]

    training_labels_for_weights.extend(
        [class_number] * class_count
    )

training_labels_for_weights = np.array(
    training_labels_for_weights
)

calculated_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=training_labels_for_weights
)

CLASS_WEIGHTS = {
    0: float(calculated_weights[0]),
    1: float(calculated_weights[1])
}

print("Calculated class weights:")
print(CLASS_WEIGHTS)

Calculated class weights:
{0: 0.820906432748538, 1: 1.2790432801822325}


In [14]:
def create_datasets(batch_size):
    training_dataset = (
        tf.keras.utils
        .image_dataset_from_directory(
            TRAIN_DIRECTORY,
            class_names=CLASS_NAMES,
            label_mode="binary",
            image_size=IMAGE_SIZE,
            batch_size=batch_size,
            shuffle=True,
            seed=SEED
        )
    )

    validation_dataset = (
        tf.keras.utils
        .image_dataset_from_directory(
            VALIDATION_DIRECTORY,
            class_names=CLASS_NAMES,
            label_mode="binary",
            image_size=IMAGE_SIZE,
            batch_size=batch_size,
            shuffle=False
        )
    )

    automatic_tuning = tf.data.AUTOTUNE

    training_dataset = (
        training_dataset.prefetch(
            buffer_size=automatic_tuning
        )
    )

    validation_dataset = (
        validation_dataset.prefetch(
            buffer_size=automatic_tuning
        )
    )

    return (
        training_dataset,
        validation_dataset
    )

In [15]:
test_training_dataset, test_validation_dataset = (
    create_datasets(batch_size=16)
)

for images, labels in (
    test_training_dataset.take(1)
):
    print("Image batch shape:")
    print(images.shape)

    print("\nLabel batch shape:")
    print(labels.shape)

    print("\nPixel value range:")
    print(
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )

Found 1123 files belonging to 2 classes.
Found 361 files belonging to 2 classes.
Image batch shape:
(16, 224, 224, 3)

Label batch shape:
(16, 1)

Pixel value range:
0.0 to 255.0


In [16]:
del test_training_dataset
del test_validation_dataset

gc.collect()

459

In [17]:
def create_data_augmentation():
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip(
                mode="horizontal"
            ),

            tf.keras.layers.RandomRotation(
                factor=0.05
            ),

            tf.keras.layers.RandomZoom(
                height_factor=0.10,
                width_factor=0.10
            ),

            tf.keras.layers.RandomContrast(
                factor=0.10
            )
        ],
        name="data_augmentation"
    )

In [18]:
def create_training_metrics():
    return [
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="roc_auc",
            curve="ROC"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]

In [19]:
def create_callbacks(
    model_path,
    history_name
):
    model_checkpoint = (
        tf.keras.callbacks.ModelCheckpoint(
            filepath=model_path,
            monitor="val_pr_auc",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    )

    early_stopping = (
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=5,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        )
    )

    reduce_learning_rate = (
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=0.0000001,
            verbose=1
        )
    )

    comma_separated_value_logger = (
        tf.keras.callbacks.CSVLogger(
            filename=(
                HISTORY_DIRECTORY
                / f"{history_name}.csv"
            ),
            append=False
        )
    )

    return [
        model_checkpoint,
        early_stopping,
        reduce_learning_rate,
        comma_separated_value_logger
    ]

PART A — Tuned Custom Convolutional Neural Network

In [20]:
def build_tuned_custom_cnn():
    inputs = tf.keras.Input(
        shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            3
        ),
        name="input_image"
    )

    x = create_data_augmentation()(
        inputs
    )

    x = tf.keras.layers.Rescaling(
        scale=1.0 / 255.0,
        name="pixel_rescaling"
    )(x)

    x = tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(
        filters=128,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(
        filters=256,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(
        units=128,
        activation="relu",
        kernel_regularizer=(
            tf.keras.regularizers.l2(
                L2_REGULARISATION
            )
        )
    )(x)

    x = tf.keras.layers.Dropout(
        rate=CUSTOM_CNN_DROPOUT
    )(x)

    outputs = tf.keras.layers.Dense(
        units=1,
        activation="sigmoid",
        name="fall_probability"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="tuned_custom_cnn"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=(
                CUSTOM_CNN_LEARNING_RATE
            )
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=create_training_metrics()
    )

    return model

In [21]:
tf.keras.backend.clear_session()
gc.collect()

custom_training_dataset, custom_validation_dataset = (
    create_datasets(
        batch_size=CUSTOM_CNN_BATCH_SIZE
    )
)

tuned_custom_cnn = (
    build_tuned_custom_cnn()
)

tuned_custom_cnn.summary()


Found 1123 files belonging to 2 classes.
Found 361 files belonging to 2 classes.


Model: "tuned_custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pixel_rescaling (Rescaling)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 422,881 (1.61 MB)

 Trainable params: 421,921 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [23]:
from pathlib import Path

PROJECT_ROOT = Path("..")

MODELS_DIRECTORY = PROJECT_ROOT / "models"

OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "hyperparameter_training"
)

HISTORY_DIRECTORY = (
    PROJECT_ROOT
    / "histories"
    / "hyperparameter_training"
)

# Create all required folders
MODELS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

HISTORY_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("Models folder:")
print(MODELS_DIRECTORY.resolve())
print("Exists:", MODELS_DIRECTORY.exists())

print("\nOutput folder:")
print(OUTPUT_DIRECTORY.resolve())
print("Exists:", OUTPUT_DIRECTORY.exists())

print("\nHistory folder:")
print(HISTORY_DIRECTORY.resolve())
print("Exists:", HISTORY_DIRECTORY.exists())

Models folder:
C:\Users\GArvit\Desktop\Fall Detection\models
Exists: True

Output folder:
C:\Users\GArvit\Desktop\Fall Detection\outputs\hyperparameter_training
Exists: True

History folder:
C:\Users\GArvit\Desktop\Fall Detection\histories\hyperparameter_training
Exists: True


In [24]:
def create_callbacks(model_path, history_name):

    # Ensure the folders exist whenever callbacks are created
    Path(model_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    HISTORY_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True
    )

    model_checkpoint = (
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_pr_auc",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    )

    early_stopping = (
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=5,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        )
    )

    reduce_learning_rate = (
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=0.0000001,
            verbose=1
        )
    )

    history_file_path = (
        HISTORY_DIRECTORY
        / f"{history_name}.csv"
    )

    csv_logger = tf.keras.callbacks.CSVLogger(
        filename=str(history_file_path),
        append=False
    )

    return [
        model_checkpoint,
        early_stopping,
        reduce_learning_rate,
        csv_logger
    ]

In [25]:
CUSTOM_CNN_MODEL_PATH = (
    MODELS_DIRECTORY
    / "tuned_custom_cnn.keras"
)

CUSTOM_CNN_HISTORY_PATH = (
    HISTORY_DIRECTORY
    / "tuned_custom_cnn.csv"
)

print("Model will be saved at:")
print(CUSTOM_CNN_MODEL_PATH.resolve())

print("\nHistory will be saved at:")
print(CUSTOM_CNN_HISTORY_PATH.resolve())

print("\nModel parent folder exists:")
print(CUSTOM_CNN_MODEL_PATH.parent.exists())

print("\nHistory parent folder exists:")
print(CUSTOM_CNN_HISTORY_PATH.parent.exists())

Model will be saved at:
C:\Users\GArvit\Desktop\Fall Detection\models\tuned_custom_cnn.keras

History will be saved at:
C:\Users\GArvit\Desktop\Fall Detection\histories\hyperparameter_training\tuned_custom_cnn.csv

Model parent folder exists:
True

History parent folder exists:
True


In [26]:
custom_cnn_history = tuned_custom_cnn.fit(
    custom_training_dataset,
    validation_data=custom_validation_dataset,
    epochs=CUSTOM_CNN_MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=create_callbacks(
        model_path=CUSTOM_CNN_MODEL_PATH,
        history_name="tuned_custom_cnn"
    )
)

Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6581 - loss: 0.6593 - pr_auc: 0.6007 - precision: 0.5505 - recall: 0.6834 - roc_auc: 0.7173
Epoch 1: val_pr_auc improved from None to 0.41850, saving model to ..\models\tuned_custom_cnn.keras

Epoch 1: finished saving model to ..\models\tuned_custom_cnn.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.6581 - loss: 0.6593 - pr_auc: 0.6007 - precision: 0.5505 - recall: 0.6834 - roc_auc: 0.7173 - val_accuracy: 0.5928 - val_loss: 0.6937 - val_pr_auc: 0.4185 - val_precision: 0.4902 - val_recall: 0.1712 - val_roc_auc: 0.5117 - learning_rate: 0.0010
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7124 - loss: 0.5855 - pr_auc: 0.6689 - precision: 0.6082 - recall: 0.7426 - roc_auc: 0.7805
Epoch 2: val_pr_auc improved from 0.41850 to 0.47327, saving model to ..\models\tuned_custom_cnn.keras

Epoch 2: finished saving model to ..\models\tuned_custom_cnn.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0

In [27]:
print(CUSTOM_CNN_MODEL_PATH.exists())
print(CUSTOM_CNN_MODEL_PATH.resolve())

True
C:\Users\GArvit\Desktop\Fall Detection\models\tuned_custom_cnn.keras


In [28]:
best_epoch_index = np.argmax(
    custom_cnn_history.history["val_pr_auc"]
)

print("Best epoch:", best_epoch_index + 1)

for metric_name in [
    "val_accuracy",
    "val_precision",
    "val_recall",
    "val_roc_auc",
    "val_pr_auc",
    "val_loss"
]:
    print(
        metric_name,
        custom_cnn_history.history[metric_name][best_epoch_index]
    )

Best epoch: 2
val_accuracy 0.5844875574111938
val_precision 0.4878048896789551
val_recall 0.5479452013969421
val_roc_auc 0.5965275168418884
val_pr_auc 0.47326791286468506
val_loss 0.6942703723907471


PART B — Tuned MobileNetV3

In [30]:
del tuned_custom_cnn
del custom_training_dataset
del custom_validation_dataset

tf.keras.backend.clear_session()
gc.collect()

print("Memory cleared.")

Memory cleared.


In [31]:
def build_tuned_mobilenetv3():
    inputs = tf.keras.Input(
        shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            3
        ),
        name="input_image"
    )

    x = create_data_augmentation()(inputs)

    backbone = tf.keras.applications.MobileNetV3Small(
        input_shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            3
        ),
        include_top=False,
        weights="imagenet",
        include_preprocessing=True
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False
    )

    x = tf.keras.layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = tf.keras.layers.Dense(
        units=128,
        activation="relu",
        kernel_regularizer=(
            tf.keras.regularizers.l2(
                L2_REGULARISATION
            )
        )
    )(x)

    x = tf.keras.layers.Dropout(
        rate=MOBILENET_DROPOUT
    )(x)

    outputs = tf.keras.layers.Dense(
        units=1,
        activation="sigmoid",
        name="fall_probability"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="tuned_mobilenetv3"
    )

    return model, backbone

In [32]:
tf.keras.backend.clear_session()
gc.collect()

mobilenet_training_dataset, mobilenet_validation_dataset = (
    create_datasets(
        batch_size=MOBILENET_BATCH_SIZE
    )
)

tuned_mobilenetv3, mobilenet_backbone = (
    build_tuned_mobilenetv3()
)

tuned_mobilenetv3.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=(
            TRANSFER_HEAD_LEARNING_RATE
        )
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=create_training_metrics()
)

tuned_mobilenetv3.summary()

Found 1123 files belonging to 2 classes.
Found 361 files belonging to 2 classes.


Model: "tuned_mobilenetv3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fall_probability (Dense)        │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,013,105 (3.86 MB)

 Trainable params: 73,985 (289.00 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [33]:
MOBILENET_PHASE1_PATH = (
    MODELS_DIRECTORY
    / "tuned_mobilenetv3_phase1.keras"
)

mobilenet_phase1_history = tuned_mobilenetv3.fit(
    mobilenet_training_dataset,
    validation_data=mobilenet_validation_dataset,
    epochs=FROZEN_BACKBONE_MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=create_callbacks(
        model_path=MOBILENET_PHASE1_PATH,
        history_name="tuned_mobilenetv3_phase1"
    )
)

Epoch 1/15
35/36 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.7741 - loss: 0.5195 - pr_auc: 0.7181 - precision: 0.6756 - recall: 0.8101 - roc_auc: 0.8380
Epoch 1: val_pr_auc improved from None to 0.71184, saving model to ..\models\tuned_mobilenetv3_phase1.keras

Epoch 1: finished saving model to ..\models\tuned_mobilenetv3_phase1.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 178ms/step - accuracy: 0.7747 - loss: 0.5187 - pr_auc: 0.7193 - precision: 0.6768 - recall: 0.8109 - roc_auc: 0.8385 - val_accuracy: 0.7091 - val_loss: 0.5744 - val_pr_auc: 0.7118 - val_precision: 0.5981 - val_recall: 0.8562 - val_roc_auc: 0.8079 - learning_rate: 0.0010
Epoch 2/15
35/36 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.8518 - loss: 0.3664 - pr_auc: 0.8531 - precision: 0.7625 - recall: 0.9018 - roc_auc: 0.9212
Epoch 2: val_pr_auc improved from 0.71184 to 0.74031, saving model to ..\models\tuned_mobilenetv3_phase1.keras

Epoch 2: finished saving model to ..\models\tuned_mobilenetv3_phase1.keras
36/36 ━━━━━

In [34]:
mobilenet_phase1_completed_epochs = len(
    mobilenet_phase1_history.epoch
)

print(
    "MobileNetV3 Phase 1 completed epochs:",
    mobilenet_phase1_completed_epochs
)

MobileNetV3 Phase 1 completed epochs: 15


In [35]:
mobilenet_backbone.trainable = True

number_of_layers_to_fine_tune = 30

fine_tuning_start_index = (
    len(mobilenet_backbone.layers)
    - number_of_layers_to_fine_tune
)

for layer in mobilenet_backbone.layers[
    :fine_tuning_start_index
]:
    layer.trainable = False

for layer in mobilenet_backbone.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

print(
    "Total backbone layers:",
    len(mobilenet_backbone.layers)
)

print(
    "Trainable backbone layers:",
    sum(
        layer.trainable
        for layer in mobilenet_backbone.layers
    )
)

Total backbone layers: 157
Trainable backbone layers: 24


In [36]:
tuned_mobilenetv3.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=(
            FINE_TUNING_LEARNING_RATE
        )
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=create_training_metrics()
)

In [37]:
MOBILENET_FINAL_PATH = (
    MODELS_DIRECTORY
    / "tuned_mobilenetv3.keras"
)

mobilenet_phase2_final_epoch = (
    mobilenet_phase1_completed_epochs
    + FINE_TUNING_MAX_EPOCHS
)

mobilenet_phase2_history = tuned_mobilenetv3.fit(
    mobilenet_training_dataset,
    validation_data=mobilenet_validation_dataset,
    initial_epoch=mobilenet_phase1_completed_epochs,
    epochs=mobilenet_phase2_final_epoch,
    class_weight=CLASS_WEIGHTS,
    callbacks=create_callbacks(
        model_path=MOBILENET_FINAL_PATH,
        history_name="tuned_mobilenetv3_phase2"
    )
)

Epoch 16/30
35/36 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.9286 - loss: 0.2144 - pr_auc: 0.9589 - precision: 0.8714 - recall: 0.9589 - roc_auc: 0.9771
Epoch 16: val_pr_auc improved from None to 0.81557, saving model to ..\models\tuned_mobilenetv3.keras

Epoch 16: finished saving model to ..\models\tuned_mobilenetv3.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 222ms/step - accuracy: 0.9279 - loss: 0.2161 - pr_auc: 0.9575 - precision: 0.8698 - recall: 0.9590 - roc_auc: 0.9764 - val_accuracy: 0.7618 - val_loss: 0.5096 - val_pr_auc: 0.8156 - val_precision: 0.6531 - val_recall: 0.8767 - val_roc_auc: 0.8813 - learning_rate: 1.0000e-05
Epoch 17/30
35/36 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.9205 - loss: 0.2201 - pr_auc: 0.9618 - precision: 0.8583 - recall: 0.9543 - roc_auc: 0.9763
Epoch 17: val_pr_auc improved from 0.81557 to 0.81972, saving model to ..\models\tuned_mobilenetv3.keras

Epoch 17: finished saving model to ..\models\tuned_mobilenetv3.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s

In [38]:
print(
    "Final MobileNetV3 saved:",
    MOBILENET_FINAL_PATH.exists()
)

print(
    MOBILENET_FINAL_PATH.resolve()
)

Final MobileNetV3 saved: True
C:\Users\GArvit\Desktop\Fall Detection\models\tuned_mobilenetv3.keras


PART C - TUNED RESNET50

In [39]:
del tuned_mobilenetv3
del mobilenet_backbone
del mobilenet_training_dataset
del mobilenet_validation_dataset

tf.keras.backend.clear_session()
gc.collect()

print("Memory cleared.")

Memory cleared.


In [40]:
@tf.keras.utils.register_keras_serializable(
    package="FallDetection"
)
class ResNet50Preprocessing(tf.keras.layers.Layer):

    def call(self, inputs):
        return tf.keras.applications.resnet50.preprocess_input(
            inputs
        )

    def get_config(self):
        return super().get_config()

In [41]:
def build_tuned_resnet50():

    inputs = tf.keras.Input(
        shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            3
        ),
        name="input_image"
    )

    x = create_data_augmentation()(inputs)

    x = ResNet50Preprocessing(
        name="resnet50_preprocessing"
    )(x)

    backbone = tf.keras.applications.ResNet50(
        input_shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            3
        ),
        include_top=False,
        weights="imagenet"
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False
    )

    x = tf.keras.layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = tf.keras.layers.Dense(
        units=256,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(
            L2_REGULARISATION
        )
    )(x)

    x = tf.keras.layers.Dropout(
        rate=RESNET_DROPOUT
    )(x)

    outputs = tf.keras.layers.Dense(
        units=1,
        activation="sigmoid",
        name="fall_probability"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="tuned_resnet50"
    )

    return model, backbone

In [42]:
tf.keras.backend.clear_session()
gc.collect()

resnet_training_dataset, resnet_validation_dataset = (
    create_datasets(
        batch_size=RESNET_BATCH_SIZE
    )
)

tuned_resnet50, resnet_backbone = (
    build_tuned_resnet50()
)

tuned_resnet50.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=TRANSFER_HEAD_LEARNING_RATE
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=create_training_metrics()
)

tuned_resnet50.summary()

Found 1123 files belonging to 2 classes.
Found 361 files belonging to 2 classes.


Model: "tuned_resnet50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50_preprocessing          │ (None, 224, 224, 3)    │             0 │
│ (ResNet50Preprocessing)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fall_probability (Dense)        │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 524,801 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [43]:
RESNET_PHASE1_PATH = (
    MODELS_DIRECTORY
    / "tuned_resnet50_phase1.keras"
)

resnet_phase1_history = tuned_resnet50.fit(
    resnet_training_dataset,
    validation_data=resnet_validation_dataset,
    epochs=FROZEN_BACKBONE_MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=create_callbacks(
        model_path=RESNET_PHASE1_PATH,
        history_name="tuned_resnet50_phase1"
    )
)

Epoch 1/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - accuracy: 0.7898 - loss: 0.5595 - pr_auc: 0.8059 - precision: 0.7018 - recall: 0.8041 - roc_auc: 0.8729
Epoch 1: val_pr_auc improved from None to 0.91952, saving model to ..\models\tuned_resnet50_phase1.keras

Epoch 1: finished saving model to ..\models\tuned_resnet50_phase1.keras
71/71 ━━━━━━━━━━━━━━━━━━━━ 52s 678ms/step - accuracy: 0.7898 - loss: 0.5595 - pr_auc: 0.8059 - precision: 0.7018 - recall: 0.8041 - roc_auc: 0.8729 - val_accuracy: 0.7673 - val_loss: 0.4186 - val_pr_auc: 0.9195 - val_precision: 0.9306 - val_recall: 0.4589 - val_roc_auc: 0.9528 - learning_rate: 0.0010
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step - accuracy: 0.8967 - loss: 0.3013 - pr_auc: 0.9232 - precision: 0.8372 - recall: 0.9134 - roc_auc: 0.9574
Epoch 2: val_pr_auc improved from 0.91952 to 0.94542, saving model to ..\models\tuned_resnet50_phase1.keras

Epoch 2: finished saving model to ..\models\tuned_resnet50_phase1.keras
71/71 ━━━━━━━━━━━━━━━━━

In [44]:
resnet_phase1_completed_epochs = len(
    resnet_phase1_history.epoch
)

print(
    "ResNet50 Phase 1 completed epochs:",
    resnet_phase1_completed_epochs
)

ResNet50 Phase 1 completed epochs: 15


In [45]:
resnet_backbone.trainable = True

number_of_layers_to_fine_tune = 30

fine_tuning_start_index = (
    len(resnet_backbone.layers)
    - number_of_layers_to_fine_tune
)

for layer in resnet_backbone.layers[
    :fine_tuning_start_index
]:
    layer.trainable = False

for layer in resnet_backbone.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

print(
    "Total ResNet50 backbone layers:",
    len(resnet_backbone.layers)
)

print(
    "Trainable backbone layers:",
    sum(
        layer.trainable
        for layer in resnet_backbone.layers
    )
)

Total ResNet50 backbone layers: 175
Trainable backbone layers: 21


In [46]:
tuned_resnet50.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FINE_TUNING_LEARNING_RATE
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=create_training_metrics()
)

In [47]:
RESNET_FINAL_PATH = (
    MODELS_DIRECTORY
    / "tuned_resnet50.keras"
)

resnet_phase2_final_epoch = (
    resnet_phase1_completed_epochs
    + FINE_TUNING_MAX_EPOCHS
)

resnet_phase2_history = tuned_resnet50.fit(
    resnet_training_dataset,
    validation_data=resnet_validation_dataset,
    initial_epoch=resnet_phase1_completed_epochs,
    epochs=resnet_phase2_final_epoch,
    class_weight=CLASS_WEIGHTS,
    callbacks=create_callbacks(
        model_path=RESNET_FINAL_PATH,
        history_name="tuned_resnet50_phase2"
    )
)

Epoch 16/30


c:\Users\GArvit\Desktop\Fall Detection\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 572ms/step - accuracy: 0.9528 - loss: 0.1414 - pr_auc: 0.9891 - precision: 0.9251 - recall: 0.9567 - roc_auc: 0.9930
Epoch 16: val_pr_auc improved from None to 0.97975, saving model to ..\models\tuned_resnet50.keras

Epoch 16: finished saving model to ..\models\tuned_resnet50.keras
71/71 ━━━━━━━━━━━━━━━━━━━━ 58s 741ms/step - accuracy: 0.9528 - loss: 0.1414 - pr_auc: 0.9891 - precision: 0.9251 - recall: 0.9567 - roc_auc: 0.9930 - val_accuracy: 0.9501 - val_loss: 0.1883 - val_pr_auc: 0.9798 - val_precision: 0.9103 - val_recall: 0.9726 - val_roc_auc: 0.9865 - learning_rate: 1.0000e-05
Epoch 17/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 574ms/step - accuracy: 0.9715 - loss: 0.1108 - pr_auc: 0.9952 - precision: 0.9512 - recall: 0.9772 - roc_auc: 0.9968
Epoch 17: val_pr_auc improved from 0.97975 to 0.98522, saving model to ..\models\tuned_resnet50.keras

Epoch 17: finished saving model to ..\models\tuned_resnet50.keras
71/71 ━━━━━━━━━━━━━━━━━━━━ 51s 724ms/step - accuracy:

In [48]:
print(
    "Final ResNet50 saved:",
    RESNET_FINAL_PATH.exists()
)

print(
    RESNET_FINAL_PATH.resolve()
)

Final ResNet50 saved: True
C:\Users\GArvit\Desktop\Fall Detection\models\tuned_resnet50.keras
